In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [3]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': 'Do you want to hear a joke about pizza?\n\nNever mind, it’s far too **cheesy**!',
   'extras': {'signature': 'Es0dCsodARFNMg9eAmvgRTe6YaOUI0J3ySyDQgCls9Ciiciz0yqdvMrmQxzBTb9Ju4AC0q1+BalSUQW+KVAPdYyZSFpUifV1EGd+tJVw7QRu9hMEbYU/wLjkY2AQ9MCNWzu7oiEFSidoXbewy9oCZXsoTk6ZvAUlWPXWsujZ/9GQ48Wv5hESyTmhxr1PuqviLz7xmkqA3bSMCt3ES0Z0wuOt7ObIbyRLesuq1AyZIxiTHdVcwf2Wlgo+kmLz/O9pgxGh1Ij5bV1blo5CKcIAM2h4wLVmzt8BYRTokZhza76VYkxgGjHEFwIYEH3owjbVAN2eLw8Df//CdoQMe8zYf0DKrDawgBe8HHCp8ZepzK9tknEJOkEtVhJyOip2woPVGSUFGEkyZD8SxhdqXgVaxeetlNdQLI7BQz5qZYWXvSuDmCzJ/KmvkwpEkB7m0rRJ9W26VbrfIWmEw06KdW2GRfhZVUXuASOXUhaFbijS3v4BVVODrjfGWCTrjISKvM9j8lb+zeTseQh6oXhT5p8icbTn3NQAWi+MTS9vdL6eEXvzfihRwfSGdDIdyS8RGRhPoU2W0xB33V6rfDDo0hYL9YFqyY/UVXLT/ajPsqczaHexD8VlD+ocsVXWklGgj0CJIDk1t8LtHEL5iaazueBK1Ud/4MleG0BvwWKeR7jZPLPTWPQf6u6rzTrOqdkeadIRG9up+TXyg43zvr5dW88fpo7oRYgvA3nbJ2XOsO8czE9PoZeRQEQoOKuV7Hij+AGzdPJFaJm12uLKEdky9SoiRhORoMfCAPwRbP4c24fiQNXDuQov+PrsNLL7IdZSobr7

In [4]:
workflow.get_state(config1)#we get the final state value of the workflow

StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Do you want to hear a joke about pizza?\n\nNever mind, it’s far too **cheesy**!', 'extras': {'signature': 'Es0dCsodARFNMg9eAmvgRTe6YaOUI0J3ySyDQgCls9Ciiciz0yqdvMrmQxzBTb9Ju4AC0q1+BalSUQW+KVAPdYyZSFpUifV1EGd+tJVw7QRu9hMEbYU/wLjkY2AQ9MCNWzu7oiEFSidoXbewy9oCZXsoTk6ZvAUlWPXWsujZ/9GQ48Wv5hESyTmhxr1PuqviLz7xmkqA3bSMCt3ES0Z0wuOt7ObIbyRLesuq1AyZIxiTHdVcwf2Wlgo+kmLz/O9pgxGh1Ij5bV1blo5CKcIAM2h4wLVmzt8BYRTokZhza76VYkxgGjHEFwIYEH3owjbVAN2eLw8Df//CdoQMe8zYf0DKrDawgBe8HHCp8ZepzK9tknEJOkEtVhJyOip2woPVGSUFGEkyZD8SxhdqXgVaxeetlNdQLI7BQz5qZYWXvSuDmCzJ/KmvkwpEkB7m0rRJ9W26VbrfIWmEw06KdW2GRfhZVUXuASOXUhaFbijS3v4BVVODrjfGWCTrjISKvM9j8lb+zeTseQh6oXhT5p8icbTn3NQAWi+MTS9vdL6eEXvzfihRwfSGdDIdyS8RGRhPoU2W0xB33V6rfDDo0hYL9YFqyY/UVXLT/ajPsqczaHexD8VlD+ocsVXWklGgj0CJIDk1t8LtHEL5iaazueBK1Ud/4MleG0BvwWKeR7jZPLPTWPQf6u6rzTrOqdkeadIRG9up+TXyg43zvr5dW88fpo7oRYgvA3nbJ2XOsO8czE9PoZeRQEQoOKuV7Hij+AGzdPJFaJm12uLKEdky9SoiRhORoMfCAPwRbP4c24fiQNXDuQov+P

In [5]:
list(workflow.get_state_history(config1))#to get intermediate values
#4 nodes-4 state values


[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Do you want to hear a joke about pizza?\n\nNever mind, it’s far too **cheesy**!', 'extras': {'signature': 'Es0dCsodARFNMg9eAmvgRTe6YaOUI0J3ySyDQgCls9Ciiciz0yqdvMrmQxzBTb9Ju4AC0q1+BalSUQW+KVAPdYyZSFpUifV1EGd+tJVw7QRu9hMEbYU/wLjkY2AQ9MCNWzu7oiEFSidoXbewy9oCZXsoTk6ZvAUlWPXWsujZ/9GQ48Wv5hESyTmhxr1PuqviLz7xmkqA3bSMCt3ES0Z0wuOt7ObIbyRLesuq1AyZIxiTHdVcwf2Wlgo+kmLz/O9pgxGh1Ij5bV1blo5CKcIAM2h4wLVmzt8BYRTokZhza76VYkxgGjHEFwIYEH3owjbVAN2eLw8Df//CdoQMe8zYf0DKrDawgBe8HHCp8ZepzK9tknEJOkEtVhJyOip2woPVGSUFGEkyZD8SxhdqXgVaxeetlNdQLI7BQz5qZYWXvSuDmCzJ/KmvkwpEkB7m0rRJ9W26VbrfIWmEw06KdW2GRfhZVUXuASOXUhaFbijS3v4BVVODrjfGWCTrjISKvM9j8lb+zeTseQh6oXhT5p8icbTn3NQAWi+MTS9vdL6eEXvzfihRwfSGdDIdyS8RGRhPoU2W0xB33V6rfDDo0hYL9YFqyY/UVXLT/ajPsqczaHexD8VlD+ocsVXWklGgj0CJIDk1t8LtHEL5iaazueBK1Ud/4MleG0BvwWKeR7jZPLPTWPQf6u6rzTrOqdkeadIRG9up+TXyg43zvr5dW88fpo7oRYgvA3nbJ2XOsO8czE9PoZeRQEQoOKuV7Hij+AGzdPJFaJm12uLKEdky9SoiRhORoMfCAPwRbP4c24fiQNXDuQov+

In [6]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': [{'type': 'text',
   'text': 'Why did the spaghetti break up with the macaroni?\n\nBecause she was too saucy, and he was too cheesy!',
   'extras': {'signature': 'EvQXCvEXARFNMg/aYrhRlyKbBf5GDTtMf0xy39oGYiacnm0J/zaTRZJ5b6XeSYbqo5MKh+0qofvqRD+qTkC/afHrI6hu1yOTlcgmaHkIiJ0cEfa3X5gRUFvu6nfLu1ZaAyD1FgvpBVkiWAwSoKM+jpxM1eNgQrqAmB4ZBG7LJZoe/WYvPSjC3+2hIiNTRnosgD13+5B5WiCWlvr4Y8vYo46DiCh5gxLfxTVg0AogcsHVLaUX/Eiir6tGL0UGVVom826Corzqo/d0JPkpmNwCy0JTJOansG9jO7Ds41oKhNb3CInvYlU3KtDFbhwm+D9CIchfYP2y+TGNjE2qeeo2TPb/s1xy9x+FhT3K+pCn5+HQhU46WbqlTT6p/IF5rRTve1QfYfiY5HCW5GtKFKxwJj4Uym2Rp0odMVpt81CFnNe2vP7XWflqKr7O/UTs6F0JN+9FiZ3vYgGMm9yvow3lY4ze9zuM8H+lydqNo/qUvnjYHL5n2+/MleiaETe6j4z2KQQwdkK9ygOubM8BNYzZIkzR0mEz+oohUsXsSM+OwQiG1c/XZvmwStL/2vWQ7ObVrHmRDI+V2kOxwTzxHahH7hRzDG8LvDr4hwHzJ34ikygCH77BiCh0l40CUx28gfMhEei/KtRBKZYya3psSK9Ukjy9+Z228OTy0PcIKDnyiFHwPq40kYNlE/NpZLKdRWKdtxAbpltYRkejUGeeaUWyI9k6z7GvG8IyCYWZqm0h684xW/0UGVbvcAFfp4tC2LUlnI/BADJPpuY4qtBc44/wZ7JIGJO9806EFmTgt34jB

In [7]:
list(workflow.get_state_history(config2))#to get intermediate values
#4 nodes-4 state values


[StateSnapshot(values={'topic': 'pasta', 'joke': [{'type': 'text', 'text': 'Why did the spaghetti break up with the macaroni?\n\nBecause she was too saucy, and he was too cheesy!', 'extras': {'signature': 'EvQXCvEXARFNMg/aYrhRlyKbBf5GDTtMf0xy39oGYiacnm0J/zaTRZJ5b6XeSYbqo5MKh+0qofvqRD+qTkC/afHrI6hu1yOTlcgmaHkIiJ0cEfa3X5gRUFvu6nfLu1ZaAyD1FgvpBVkiWAwSoKM+jpxM1eNgQrqAmB4ZBG7LJZoe/WYvPSjC3+2hIiNTRnosgD13+5B5WiCWlvr4Y8vYo46DiCh5gxLfxTVg0AogcsHVLaUX/Eiir6tGL0UGVVom826Corzqo/d0JPkpmNwCy0JTJOansG9jO7Ds41oKhNb3CInvYlU3KtDFbhwm+D9CIchfYP2y+TGNjE2qeeo2TPb/s1xy9x+FhT3K+pCn5+HQhU46WbqlTT6p/IF5rRTve1QfYfiY5HCW5GtKFKxwJj4Uym2Rp0odMVpt81CFnNe2vP7XWflqKr7O/UTs6F0JN+9FiZ3vYgGMm9yvow3lY4ze9zuM8H+lydqNo/qUvnjYHL5n2+/MleiaETe6j4z2KQQwdkK9ygOubM8BNYzZIkzR0mEz+oohUsXsSM+OwQiG1c/XZvmwStL/2vWQ7ObVrHmRDI+V2kOxwTzxHahH7hRzDG8LvDr4hwHzJ34ikygCH77BiCh0l40CUx28gfMhEei/KtRBKZYya3psSK9Ukjy9+Z228OTy0PcIKDnyiFHwPq40kYNlE/NpZLKdRWKdtxAbpltYRkejUGeeaUWyI9k6z7GvG8IyCYWZqm0h684xW/0UGVbvcAFfp4tC2LUlnI/BADJPpuY4qtBc44/wZ7JIGJ

# Time Travel

In [16]:
#each checkpoint will associated with its own  id
workflow.get_state({'configurable':{'thread_id':'1',"checkpoint_id":'1f196d8a-5358-6ea1-8000-732a7ee81603'}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f196d8a-5358-6ea1-8000-732a7ee81603'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-08-13T05:34:25.404848+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f196d8a-534e-6cf3-bfff-d5da4cc770da'}}, tasks=(PregelTask(id='33b9a397-a9df-8342-400d-4bc7480ec6b7', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': [{'type': 'text', 'text': 'Do you want to hear a joke about pizza?\n\nNever mind, it’s far too **cheesy**!', 'extras': {'signature': 'Es0dCsodARFNMg9eAmvgRTe6YaOUI0J3ySyDQgCls9Ciiciz0yqdvMrmQxzBTb9Ju4AC0q1+BalSUQW+KVAPdYyZSFpUifV1EGd+tJVw7QRu9hMEbYU/wLjkY2AQ9MCNWzu7oiEFSidoXbewy9oCZXsoTk6ZvAUlWPXWsujZ/9GQ48Wv5hESyTmhxr1PuqviLz7xmkqA3bSMCt3ES0Z0wuOt7ObIbyRLesuq1AyZIxiTHdVcwf2Wlgo+kmLz/O9pgxGh1Ij5bV1blo5CKcIAM2h4wLVmzt8

In [ ]:
workflow.invoke(None,{'configurable':{'thread_id':'1',"checkpoint_id":'1f196d8a-5358-6ea1-8000-732a7ee81603'}})
#joke changed

{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': 'What’s the difference between a good pizza joke and a bad pizza joke?\n\n**The delivery.**',
   'extras': {'signature': 'EpgPCpUPARFNMg+jXfqvw80H0BvY9zPvmKnuvJEazzza4+yrMatWz3i3anwcD3wUji5iKMRMNNIkxlPZG8tInJI7gcOf/fGtNtVjc1Y8whjqe5VnjHZI6Jaq+wG5cPy+nTTPWFD5nvJijYFm+an4osUuGAk5ZUQbGvq49h8n6SVbPobe/ikMbR0cLOWfCyn7GJEwj7YDuwThwIrrdTimJUI3Q08D/YkCps2UNn1dnU61LmBeKJLTB6RM4kvsT3hkMxmZ6/GQ5KyFRR9A7lgJV2fFO29uQShrDd/lNkAI74/YKA0Y0Id41oDX3/OdiY3UdyysxopS+sCZDPVo2+zKWPke7J/IxwYiJW+QlxaqZQhLQzwT3H8gtwMfuL6O62g7hPYRE5m9BahXvO48ZTiV5Hy9zqRo6fRnp4yb0tFqaXZBlLGLCU6Vfc0aanzv+mFMqmJPeRTkt/OSIBIowIM0u/p25Qk7PW9dGcdIH05vzI05m5yItcbTkwpaNJokEEeZJmITPZ4md6y8pSECYxPYH/QxDKLGm98bSHul1FmcX1NVrCyp4cq7RNmDXdK4vVmsMC5JpyKN74x50/nwbkQVGzhVzqLCeGeCBUYff77X3X2sGLFvnLEPVudaYweY4heFXhuaUpHN0dOM6wt9FL3W+pZ7+eHV6V0pyIv/51cXSAAorxRUS4pkKCGnsuXVYLFFSRbooHprgx+AagTwT4N1ggu9h8XUHbTaNCvn1052b//lUB5dew4vWrnisX2QEoyyjfe/hPHWBJn5TU/plO2NXsWx7MCCHI2acu9k4fkqcKjAYNwu6hyON

In [ ]:
list(workflow.get_state_history(config1))#to get intermediate values
#more values--nodes during time travel also changed


[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'What’s the difference between a good pizza joke and a bad pizza joke?\n\n**The delivery.**', 'extras': {'signature': 'EpgPCpUPARFNMg+jXfqvw80H0BvY9zPvmKnuvJEazzza4+yrMatWz3i3anwcD3wUji5iKMRMNNIkxlPZG8tInJI7gcOf/fGtNtVjc1Y8whjqe5VnjHZI6Jaq+wG5cPy+nTTPWFD5nvJijYFm+an4osUuGAk5ZUQbGvq49h8n6SVbPobe/ikMbR0cLOWfCyn7GJEwj7YDuwThwIrrdTimJUI3Q08D/YkCps2UNn1dnU61LmBeKJLTB6RM4kvsT3hkMxmZ6/GQ5KyFRR9A7lgJV2fFO29uQShrDd/lNkAI74/YKA0Y0Id41oDX3/OdiY3UdyysxopS+sCZDPVo2+zKWPke7J/IxwYiJW+QlxaqZQhLQzwT3H8gtwMfuL6O62g7hPYRE5m9BahXvO48ZTiV5Hy9zqRo6fRnp4yb0tFqaXZBlLGLCU6Vfc0aanzv+mFMqmJPeRTkt/OSIBIowIM0u/p25Qk7PW9dGcdIH05vzI05m5yItcbTkwpaNJokEEeZJmITPZ4md6y8pSECYxPYH/QxDKLGm98bSHul1FmcX1NVrCyp4cq7RNmDXdK4vVmsMC5JpyKN74x50/nwbkQVGzhVzqLCeGeCBUYff77X3X2sGLFvnLEPVudaYweY4heFXhuaUpHN0dOM6wt9FL3W+pZ7+eHV6V0pyIv/51cXSAAorxRUS4pkKCGnsuXVYLFFSRbooHprgx+AagTwT4N1ggu9h8XUHbTaNCvn1052b//lUB5dew4vWrnisX2QEoyyjfe/hPHWBJn5TU/plO2NXsWx7MCCHI2acu9k4f

## Update State

In [21]:
# 1. Fetch the authentic history of the thread
history = workflow.get_state_history({"configurable": {"thread_id": "1"}})

# 2. Extract the complete, system-generated config containing the hidden keys
target_config = None
for state in history:
    if state.config["configurable"]["checkpoint_id"] == "1f196d8a-5358-6ea1-8000-732a7ee81603":
        target_config = state.config
        break

# 3. Pass that valid system-generated config to update your state
if target_config:
    workflow.update_state(target_config, {"topic": "samosa"})
    print("✅ State updated successfully without errors!")
else:
    print("❌ Checkpoint ID not found in history. Check your ID string.")


✅ State updated successfully without errors!


In [22]:
# workflow.update_state({'configurable':{'thread_id':'1',"checkpoint_id":'1f196d8a-5358-6ea1-8000-732a7ee81603'}},{'topic':'samosa'})

In [23]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f196dbd-31c8-6b05-8001-18cdf4e94373'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-08-13T05:57:10.906323+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f196d8a-5358-6ea1-8000-732a7ee81603'}}, tasks=(PregelTask(id='c708a574-490b-9301-c6da-c86363dbbec0', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'What’s the difference between a good pizza joke and a bad pizza joke?\n\n**The delivery.**', 'extras': {'signature': 'EpgPCpUPARFNMg+jXfqvw80H0BvY9zPvmKnuvJEazzza4+yrMatWz3i3anwcD3wUji5iKMRMNNIkxlPZG8tInJI7gcOf/fGtNtVjc1Y8whjqe5VnjHZI6Jaq+wG5cPy+nTTPWFD5nvJijYFm+an4osUuGAk5ZUQbGvq49h8n6SVbPobe/ikMbR0cLO

In [24]:
workflow.invoke(None,{'configurable':{'thread_id':'1',"checkpoint_id":'1f196dbd-31c8-6b05-8001-18cdf4e94373'}})
#execute from the check point where toipc changed

{'topic': 'samosa',
 'joke': [{'type': 'text',
   'text': 'Why did the samosa go to therapy?\n\nBecause it was feeling a little **samosa-d** (so sad) and was afraid of cracking under pressure!',
   'extras': {'signature': 'ErklCrYlARFNMg+hVE7azCDNmvAQmFIXN+YB+Iu3ddcUDS5VrOaUap031szSQZUsvWA0ZdcfFljHfdTKVzTKqx676c2bkNDzWVU1tnL1R0fRZrM7IF11R0aKgBpQFNPIHHxUbh9Lcdo4GiH4my6+uR1PQnmxVScwxFqXErXfB0rIbxfCIF9uSWv/g2RAkvk1ZdRe5Jk2dQXLBYEakI6ksgWhgBPA6kLYrw9fNuvZVa8DWDXncujIpbU26bXZ7RMKCyNe6BplhK9tBLKANToAQPqfNoi7haRhx5CbFmxMYvpGnkiSWdoEWxETxE5/cknKvhjtAp2ONj6RiWgB63yeNqWolJJfC0fpocwqHJ++AmNymvAQFKgcyxoaPQkj5UOoySMtfLxdp/dFvO2gRYebISK3bys1lQ8vifXf3AadkAcFa6AfJ+PZoUmuJoO8ylma8174lFF9c3zXnNMtm47Y45E6j/LqyhetFe8fQzT9mAc5ZEWh3ycIUseLvlO7Zmhp453QuaMV/uu9p/kOJBwYaiVU9TFdCk1WqnLOkOLOXNs9JBR4aO+Y9K2/ZD5ISaYIRaGRvBX40Y4gv0ENlWG8A2+k5VBqtDMq/BcDhIbVaV7aPdQOIHY2S2IWHg733fTzDyXWuBrW6f1Rk5orvn3Z+vGo02raCybhxmMEVlpYi+ZUDzzRbmSvQDoRHC6JlAJP+uRywzRrkw4R4ts3P6E8I5s+auifKD5+rVlbKqNwLDKH5dKutCLdnVAbtmfmmHvbfOLd5hOa+

In [25]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': [{'type': 'text', 'text': 'Why did the samosa go to therapy?\n\nBecause it was feeling a little **samosa-d** (so sad) and was afraid of cracking under pressure!', 'extras': {'signature': 'ErklCrYlARFNMg+hVE7azCDNmvAQmFIXN+YB+Iu3ddcUDS5VrOaUap031szSQZUsvWA0ZdcfFljHfdTKVzTKqx676c2bkNDzWVU1tnL1R0fRZrM7IF11R0aKgBpQFNPIHHxUbh9Lcdo4GiH4my6+uR1PQnmxVScwxFqXErXfB0rIbxfCIF9uSWv/g2RAkvk1ZdRe5Jk2dQXLBYEakI6ksgWhgBPA6kLYrw9fNuvZVa8DWDXncujIpbU26bXZ7RMKCyNe6BplhK9tBLKANToAQPqfNoi7haRhx5CbFmxMYvpGnkiSWdoEWxETxE5/cknKvhjtAp2ONj6RiWgB63yeNqWolJJfC0fpocwqHJ++AmNymvAQFKgcyxoaPQkj5UOoySMtfLxdp/dFvO2gRYebISK3bys1lQ8vifXf3AadkAcFa6AfJ+PZoUmuJoO8ylma8174lFF9c3zXnNMtm47Y45E6j/LqyhetFe8fQzT9mAc5ZEWh3ycIUseLvlO7Zmhp453QuaMV/uu9p/kOJBwYaiVU9TFdCk1WqnLOkOLOXNs9JBR4aO+Y9K2/ZD5ISaYIRaGRvBX40Y4gv0ENlWG8A2+k5VBqtDMq/BcDhIbVaV7aPdQOIHY2S2IWHg733fTzDyXWuBrW6f1Rk5orvn3Z+vGo02raCybhxmMEVlpYi+ZUDzzRbmSvQDoRHC6JlAJP+uRywzRrkw4R4ts3P6E8I5s+auifKD5+rVlbKqNwLDKH5dKutCLdnVAbtm